In [1]:
import random

# ==========================================
# STEP 1: PROBLEM CONFIGURATION & CONSTRAINTS
# ==========================================
NUM_EVS = 3
HOURS = 5
TIME_LABELS = ["6PM-9PM", "9PM-12AM", "12AM-2AM", "2AM-4AM", "4AM-6AM"]

# Electricity price weights (3 = high cost, 1 = cheap nighttime rate)
PRICE_CURVE = [3, 2, 1, 1, 2]

# Physical engineering limits
GRID_CAPACITY = 15.0      # Maximum kW the grid can handle combined per hour
MAX_CHARGER_SPEED = 7.0   # Maximum kW a single car's charger can handle

# Battery energy requirements for [EV1, EV2, EV3] to hit 100%
EV_BATTERY_NEEDS = [9.0, 14.0, 14.0]

# PSO Hyperparameters
POPULATION_SIZE = 20
MAX_ITERATIONS = 60
W = 0.5   # Inertia (momentum)
C1 = 1.5  # Cognitive coefficient (trusting own memory)
C2 = 1.5  # Social coefficient (trusting swarm memory)

# ==========================================
# STEP 2: FITNESS FUNCTION (THE GRADER)
# ==========================================
def calculate_fitness(schedule):
    """
    Grades a schedule. Lower score is better.
    Calculates cost and slaps heavy financial penalties for breaking rules.
    """
    electricity_cost = 0.0
    grid_penalty = 0.0
    battery_penalty = 0.0

    # Track how much total power is pulled every hour
    hourly_grid_load = [0.0] * HOURS

    for ev_idx in range(NUM_EVS):
        total_charged_into_ev = 0.0
        for hour in range(HOURS):
            power_allocated = schedule[ev_idx][hour]
            hourly_grid_load[hour] += power_allocated

            # Calculate bill using the dynamic price curve
            electricity_cost += power_allocated * PRICE_CURVE[hour]
            total_charged_into_ev += power_allocated

        # RULE 1 PENALTY: Did the EV get fully charged?
        if total_charged_into_ev < EV_BATTERY_NEEDS[ev_idx]:
            # Add a massive penalty if the car is left empty
            battery_penalty += (EV_BATTERY_NEEDS[ev_idx] - total_charged_into_ev) * 100.0

    # RULE 2 PENALTY: Did we overload the local transformer grid?
    for hour in range(HOURS):
        if hourly_grid_load[hour] > GRID_CAPACITY:
            grid_penalty += (hourly_grid_load[hour] - GRID_CAPACITY) * 200.0

    return electricity_cost + battery_penalty + grid_penalty

def enforce_physical_boundaries(schedule):
    """Clamps numbers between 0 and the maximum physical charger speed."""
    for ev in range(NUM_EVS):
        for hour in range(HOURS):
            if schedule[ev][hour] < 0:
                schedule[ev][hour] = 0.0
            elif schedule[ev][hour] > MAX_CHARGER_SPEED:
                schedule[ev][hour] = MAX_CHARGER_SPEED
    return schedule

# ==========================================
# STEP 3: INITIALIZE THE SWARM
# ==========================================
# Create population array matrices
particles_position = []
particles_velocity = []
personal_best_position = []
personal_best_scores = []

for _ in range(POPULATION_SIZE):
    # Create a completely random charging grid matrix [EVs x Hours]
    random_schedule = [[random.uniform(0, 5) for _ in range(HOURS)] for _ in range(NUM_EVS)]
    random_schedule = enforce_physical_boundaries(random_schedule)

    particles_position.append(random_schedule)
    personal_best_position.append(list(random_schedule))
    personal_best_scores.append(calculate_fitness(random_schedule))

    # Give it an initial movement velocity speed
    random_velocity = [[random.uniform(-1, 1) for _ in range(HOURS)] for _ in range(NUM_EVS)]
    particles_velocity.append(random_velocity)

# Set Global Best to the best initial guess found
global_best_score = min(personal_best_scores)
global_best_position = list(particles_position[personal_best_scores.index(global_best_score)])

# ==========================================
# STEP 4: MAIN PSO OPTIMIZATION LOOP
# ==========================================
for iteration in range(MAX_ITERATIONS):
    for i in range(POPULATION_SIZE):

        # 1. Move Particle (Update Velocities & Positions hourly)
        for ev in range(NUM_EVS):
            for hour in range(HOURS):
                r1, r2 = random.random(), random.random()

                # The Core PSO Math Equation
                particles_velocity[i][ev][hour] = (
                    W * particles_velocity[i][ev][hour] +
                    C1 * r1 * (personal_best_position[i][ev][hour] - particles_position[i][ev][hour]) +
                    C2 * r2 * (global_best_position[ev][hour] - particles_position[i][ev][hour])
                )
                particles_position[i][ev][hour] += particles_velocity[i][ev][hour]

        # 2. Keep values physically realistic
        particles_position[i] = enforce_physical_boundaries(particles_position[i])

        # 3. Re-grade the schedule
        current_score = calculate_fitness(particles_position[i])

        # 4. Check if it beat its personal record
        if current_score < personal_best_scores[i]:
            personal_best_scores[i] = current_score
            personal_best_position[i] = list(particles_position[i])

        # 5. Check if it beat the entire swarm's record
        if current_score < global_best_score:
            global_best_score = current_score
            global_best_position = list(particles_position[i])

# ==========================================
# STEP 5: DISPLAY THE MATRIX OUTPUT
# ==========================================
print("\n=== OPTIMIZATION COMPLETE ===")
print(f"Final Swarm Fitness Target Score: {global_best_score:.2f}")
print(f"Target Combined Max Grid Safety Load: {GRID_CAPACITY} kW\n")

print(f"{'Time Slot':<12} | {'EV 1 (kW)':<10} | {'EV 2 (kW)':<10} | {'EV 3 (kW)':<10} | {'Total Load':<12}")
print("-" * 65)

for hour in range(HOURS):
    ev1_power = global_best_position[0][hour]
    ev2_power = global_best_position[1][hour]
    ev3_power = global_best_position[2][hour]
    total_load = ev1_power + ev2_power + ev3_power

    print(f"{TIME_LABELS[hour]:<12} | {ev1_power:<10.2f} | {ev2_power:<10.2f} | {ev3_power:<10.2f} | {total_load:<12.2f}")

print("-" * 65)
print(f"{'Total Delivered':<12} | {sum(global_best_position[0]):<10.2f} | {sum(global_best_position[1]):<10.2f} | {sum(global_best_position[2]):<10.2f} |")
print(f"{'Target Target':<12} | {EV_BATTERY_NEEDS[0]:<10.2f} | {EV_BATTERY_NEEDS[1]:<10.2f} | {EV_BATTERY_NEEDS[2]:<10.2f} | (Battery constraints met!)")



=== OPTIMIZATION COMPLETE ===
Final Swarm Fitness Target Score: 62.95
Target Combined Max Grid Safety Load: 15.0 kW

Time Slot    | EV 1 (kW)  | EV 2 (kW)  | EV 3 (kW)  | Total Load  
-----------------------------------------------------------------
6PM-9PM      | 0.00       | 4.44       | 7.00       | 11.44       
9PM-12AM     | 0.00       | 0.00       | 0.00       | 0.00        
12AM-2AM     | 2.69       | 7.00       | 0.00       | 9.69        
2AM-4AM      | 7.00       | 2.21       | 4.98       | 14.19       
4AM-6AM      | 0.00       | 0.00       | 1.38       | 1.38        
-----------------------------------------------------------------
Total Delivered | 9.69       | 13.65      | 13.36      |
Target Target | 9.00       | 14.00      | 14.00      | (Battery constraints met!)
